# BLOQUE 01 – PERFIL DEMOGRÁFICO
## Objetivo
Construir capa consolidada demográfica normalizada por unidad espacial.
## Inputs
- poblacion.shp
- genero.shp
- discapacidad.csv
## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización
## Output
1_D4C1V1_output_Movilidad_activa
- 1_D4C1V1_Concentrado_Mi_Bici.GPKG
- 1_D4C1V1_Ciclovía.GPKG


In [2]:
!pip install geopandas pyogrio

   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   ---------------------------------------  22.8/22.9 MB 113.7 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 86.4 MB/s  0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 82.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 62.9 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------- ----------------------------- 1/4 [pyproj]
   -------------------- ------------------- 2/4 [pyogrio]
   -------------------- ------------------- 2/4 [pyogrio]
   -------------------- ------------------- 2/4 [pyogrio]
   ------------------------------ --------- 3/4 [geopandas]
   ------------------------------ --------- 3/4 [geopandas]
   -----------------

In [3]:
import geopandas as gpd
import pandas as pd
import os

In [4]:
ruta_ciclovias = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\Mi Bici y Ciclovía\Ciclovías.gpkg"
ruta_estaciones = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\Mi Bici y Ciclovía\Estaciones Mi Bici.gpkg"
carpeta_salida = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\ D4C1V1_Movilidad_activa"
archivo_final = os.path.join(carpeta_salida, "1_D4C1V1_output_Movilidad_activa.gpkg")

In [12]:
if not os.path.exists(carpeta_salida):
    os.makedirs(carpeta_salida)
    print(f"Carpeta creada: {carpeta_salida}")

# --- 2. PROCESAMIENTO DE CICLOVÍAS ---
print("Procesando Ciclovías...")
gdf_ciclo = gpd.read_file(ruta_ciclovias)

# Añadir columna categoría
gdf_ciclo['Categoría'] = 'ciclovía'

# Borrar columnas específicas
cols_a_borrar_ciclo = ["ano_de_rec", "ano_de_con", "kilometros", "tipo_de_ci"]
# Usamos errors='ignore' por si alguna ya no existe o el nombre varía un poco
gdf_ciclo = gdf_ciclo.drop(columns=cols_a_borrar_ciclo, errors='ignore')

# --- 3. PROCESAMIENTO DE ESTACIONES MI BICI ---
print("Procesando Estaciones Mi Bici...")
gdf_estaciones = gpd.read_file(ruta_estaciones)

# Añadir columna categoría
gdf_estaciones['Categoría'] = 'estación_mi_bici'

# Borrar columna específica
gdf_estaciones = gdf_estaciones.drop(columns=["clave_de_i"], errors='ignore')

# --- 4. HOMOLOGAR COORDENADAS (Importante para la unión) ---
# Nos aseguramos que las estaciones tengan el mismo CRS que las ciclovías
if gdf_estaciones.crs != gdf_ciclo.crs:
    gdf_estaciones = gdf_estaciones.to_crs(gdf_ciclo.crs)

# --- 5. UNIÓN Y GUARDADO ---
print("Uniendo capas y guardando GPKG...")
# Concatenamos los dos GeoDataFrames
gdf_final = pd.concat([gdf_ciclo, gdf_estaciones], ignore_index=True)

# Convertir a GeoDataFrame final
gdf_final = gpd.GeoDataFrame(gdf_final, geometry='geometry', crs=gdf_ciclo.crs)

# Guardar en la ruta especificada
gdf_final.to_file(archivo_final, driver="GPKG")

print("-" * 30)
print(f"¡Éxito total! El archivo se guardó en:\n{archivo_final}")

Procesando Ciclovías...
Procesando Estaciones Mi Bici...
Uniendo capas y guardando GPKG...
------------------------------
¡Éxito total! El archivo se guardó en:
C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\ D4C1V1_Movilidad_activa\1_D4C1V1_output_Movilidad_activa.gpkg
